# COde 03 — Đánh giá nghiên cứu và khóa test

Gắn bank v4, cache notebook01 và output notebook02. Mặc định chỉ đánh giá validation.
Test chỉ mở sau khi hoàn thành thực nghiệm và ghi protocol đã khóa cùng các giới hạn review còn lại.
Không chỉnh phương pháp/hyperparameters theo test.

In [ ]:
from pathlib import Path
import os, sys, json
RESEARCH_BANK_OVERRIDE = None
PROJECT_ROOT_OVERRIDE = None
FEATURE_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None
PREVIOUS_STUDY_OVERRIDE = None
RUN_NAME = 'code_research_v1'
SPLIT = 'validation'
TEST_REVIEW_RECORD = None
STUDY_ROOT_OVERRIDE = None

In [ ]:
from pathlib import Path
import os, sys, json, importlib.util
RESEARCH_BANK_OVERRIDE = globals().get('RESEARCH_BANK_OVERRIDE')
RESEARCH_ASSETS = None

def research_find(predicate, *, include_runs=False):
    found = set()
    for base in (Path.cwd(), Path.cwd().parent / 'Data', Path('/kaggle/input')):
        if not base.exists():
            continue
        for current, dirs, files in os.walk(base):
            here = Path(current)
            if predicate(here, dirs, files):
                found.add(here.resolve())
            excluded = {'Images', '.git', '.cache', '__pycache__', '_archive', '_support',
                        'models', 'wheels', '_chunks', 'student_export'}
            if not include_runs:
                excluded.add('runs')
            dirs[:] = [d for d in dirs if d not in excluded]
            if len(here.relative_to(base).parts) >= 8:
                dirs[:] = []
    return sorted(found, key=str)

def research_one(override, found, name):
    if override is not None:
        return Path(override).resolve()
    if len(found) != 1:
        raise RuntimeError(f'{name}: found {len(found)} folders: {found}. Set the matching *_OVERRIDE.')
    return found[0]

if Path('/kaggle').exists():
    bank_root = research_one(RESEARCH_BANK_OVERRIDE,
        research_find(lambda p, d, f: 'research_bank_manifest.json' in f), 'Research bank')
    helper = bank_root / 'source/_support/research_bank.py'
    spec = importlib.util.spec_from_file_location('research_bank_setup', helper)
    bank = importlib.util.module_from_spec(spec); spec.loader.exec_module(bank)
    RESEARCH_ASSETS = bank.activate(bank_root)
    PROJECT_ROOT = RESEARCH_ASSETS['source_root']
    # Compatibility for existing data-audit publishing cells only.
    OFFLINE_ASSETS = {'root': bank_root, 'source_root': PROJECT_ROOT}
else:
    PROJECT_ROOT = research_one(globals().get('PROJECT_ROOT_OVERRIDE'),
        research_find(lambda p, d, f: (p / 'clindistill/research').is_dir() and 'configs' in d), 'Research source')
    sys.path.insert(0, str(PROJECT_ROOT))
print('Research source:', PROJECT_ROOT)

In [ ]:
from clindistill.artifacts import read_json
FEATURE_ROOT = research_one(FEATURE_ROOT_OVERRIDE,
    research_find(lambda p,d,f: 'feature_manifest.json' in f and
        not read_json(p / 'feature_manifest.json').get('debug') and
        {'train.npz','validation.npz'} <= set(f)), 'Research feature cache v4')
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE) if OUTPUT_ROOT_OVERRIDE else (
    Path('/kaggle/working') if Path('/kaggle/working').exists() else PROJECT_ROOT / '_support')
STUDY_ROOT = OUTPUT_ROOT / 'runs' / RUN_NAME
CONFIG = read_json(PROJECT_ROOT / 'configs/code/research.json')
CONFIG['run_name'] = RUN_NAME
print('Features:', FEATURE_ROOT, '\nStudy:', STUDY_ROOT)

In [ ]:
from clindistill.research.reporting import evaluate_study
if STUDY_ROOT_OVERRIDE:
    STUDY_ROOT = Path(STUDY_ROOT_OVERRIDE)
elif not (STUDY_ROOT / 'completed.json').exists():
    STUDY_ROOT = research_one(None, research_find(lambda p,d,f:
        {'study.json','completed.json'} <= set(f) and read_json(p / 'study.json')['config']['run_name'] == RUN_NAME,
        include_runs=True), 'Completed study')
table, summary = evaluate_study(FEATURE_ROOT, STUDY_ROOT, OUTPUT_ROOT / 'reports' / RUN_NAME / SPLIT,
                              split=SPLIT, test_review_record=TEST_REVIEW_RECORD)
display(table); display(summary)

Đầu ra: mean/std qua seeds, per-class recall/AUROC, confusion matrices, ECE/Brier, risk–coverage,
case/subgroup analysis, group-bootstrap CI và 4 so sánh đã đăng ký.
CI điều kiện trên mô hình đã fit; không thay cho biến động retraining. CI của nhiều so sánh chưa hiệu chỉnh,
chỉ diễn giải mô tả. COde internal evaluation chưa thay thế external validation.

Mở test: notebook01 tạo thêm shard test với đúng bank/policy, không đổi cache train/validation;
đặt SPLIT='test' và TEST_REVIEW_RECORD mô tả protocol đã khóa cùng vấn đề annotation/clinical còn lại.